In [1]:
import sys
import argparse
import concurrent.futures
import glob
import logging
import os
import time
import traceback
import multiprocessing as mp

from multiprocessing import Process
from multiprocessing import Queue

# set environment vars before numpy import
from marsdataio.os.setenv import set_env_vars, set_all_random_seed

set_env_vars(1)

import cv2

cv2.setNumThreads(0)

import numpy as np
import torch

from visdom import Visdom

from marsdataio.logging import init_logger, deinit_logger
from marsdataio.os.multiprocess import NoDaemonPool
from dataengine.generator.drivemode.drivemode import DriveModeGeneratorHandler
from dataengine.generator.renderer import RenderHandler
from dataengine.generator.nav.nav import NavGenerator, plot_legs_map

# from dataengine.generator.obstacle.segment_tracker import CameraTrackingHandler
from dataengine.generator.obstacle.obstacle import ObstacleGeneratorHandler
from dataengine.generator.obstacle.radar_assoc import ObstacleRadarAssocHandler
from dataengine.generator.road.road import RoadDataGeneratorHandler
from marsdataio.dbhelper import (
    CameraTopic,
    DbReader,
    extract_cam_timestamps,
    get_all_radar_configs,
    get_db_config,
    get_radar_topic,
    run_segmented_databases,
    trim_db_time,
)
from marsdataio.npyhelper import (
    get_all_cameras_params,
    get_db_and_video_paths,
    extract_nav_leg_graph,
    extract_npy,
)
from marstransform.geotrans import ecef2geodetic
from marsneuralzoo.models.drivenet import DriveNet
from marsneuralzoo.models.segnet import Segnet
from marsneuralzoo.models.yolo import Yolo
from marsneuralzoo.models.yolo_seg import SegmentationYolo
from marsneuralzoo.models.e2emvm import E2emvm


# 전달할 인자를 직접 설정
sys.argv = [
    '--n_process=1',
    '--src_base_dir',
    '/media/vol/shared/mars_dataset/ego_motion/2023_10/',
    '--output_dir',
    '/home/mars/mars_test/',
    '--npy_prefixes',
    # '2023_10_12_11_13_43_seg_8',
    # '2023_02_27_15_21_49_seg_175_seq_3',
    # '2023_11_01_07_30_47_seg_241',
    '2023_10_19_07_21_48_seg_53',
    '--s',
    '200',
    '--u',
    '10',
    '--override_output',
    '--debug',
]

# argparse 설정
parser = argparse.ArgumentParser()

parser.add_argument(
    '--vis',
    required=False,
    help='enable visualisation',
    dest='vis',
    action='store_true',
)
parser.add_argument(
    '--npy_prefixes',
    nargs='+',
    required=False,
    default=[],
    help='npy name prefixes, can be more than one, e.g., 2020_09_07 2020_09_07',
)
parser.add_argument(
    '--npy_list_file',
    required=False,
    help=(
        'text file containing list of npy files to process\n'
        'each line should contain the path to an npy file relative to '
        '--src_base_dir'
    ),
)
parser.add_argument(
    '--src_base_dir',
    required=False,
    help='source base dir',
    default='/media/vol/runs/ego_motion',
)
parser.add_argument(
    '--output_dir',
    required=False,
    help='output base path',
    default='/media/vol/runs/unified_data',
)
parser.add_argument(
    '--n_process',
    required=False,
    help='number of processes',
    default=9,
    type=int,
)
parser.add_argument(
    '--s',
    required=False,
    help='npy start time in second',
    default=0,
    type=int,
)
parser.add_argument(
    '--u',
    required=False,
    help='npy duration in second',
    default=0,
    type=int,
)
parser.add_argument(
    '--no_obstacle',
    required=False,
    help='exclude obstacles from data generation',
    dest='no_obstacle',
    action='store_true',
)
parser.add_argument(
    '--no_laneline',
    required=False,
    help='exclude laneline from data generation',
    dest='no_laneline',
    action='store_true',
)
parser.add_argument(
    '--no_drivemode',
    required=False,
    help='exclude drive mode from data generation',
    dest='no_drivemode',
    action='store_true',
)
parser.add_argument(
    '--no_nav',
    required=False,
    help='exclude navigation from data generation',
    dest='no_nav',
    action='store_true',
)
parser.add_argument(
    '--override_output',
    required=False,
    help='override the npy if exist, else skip the datagen if the npy exists',
    dest='override_output',
    action='store_true',
)
parser.add_argument(
    '--debug',
    required=False,
    help='set this flag to enable debug output',
    dest='debug',
    action='store_true',
)
parser.set_defaults(vis=False, override_output=False, debug=False)

# 인자 파싱
options = parser.parse_args()

configs = {
    'db_base_path': '/media/vol/shared/db?',
    'src_base_path': options.src_base_dir,
    'output_base_path': options.output_dir,
    'diskcache_dir': '/tmp/diskcache',
    'visualisation': options.vis,
    'remote_vis': False,
    'x_range': 200,
    'lane_model_path': '/media/vol/shared/runs/model/drivenet/drivenet_v100_286_best.pt',
    'seg_model_path': '/media/vol/shared/runs/model/segnet/comma10k/segnet_v112_60.pt',
    'obj_det_model_path': '/media/vol/shared/runs/detection_run/model/yolov3/marsdeepdrive/yolov3_v100_100_last_608.pt',
    'ego_mask_dir': '/media/vol/shared/obstacle/ego_mask/latest',
    'cuboid_prior_wlh': np.array([2.0, 4.0, 1.5]),
    'path_width': 4.0,
    'n_subprocesses': 4,
    'debug': options.debug,
    'gen_obstacle': not options.no_obstacle,
    'gen_laneline': not options.no_laneline,
    'gen_drivemode': not options.no_drivemode,
    'gen_nav': not options.no_nav,
    'osm_psql_params': {
        'dbname': 'osm',
        'user': 'osmuser',
        'password': 'osmuser',
        'host': '1.deep.local.marsauto.io',
        'port': '5432',
    },
    'valhalla_url': 'http://1.deep.local.marsauto.io:8002',
    'start_time': options.s,
    'duration': options.u,
}
# mp.set_start_method('spawn')
session = time.strftime('%Y-%m-%d_%H_%M_%S')
error_log_path = os.path.join(
    configs['output_base_path'], f'error_{session}.log'
)

# Init error logger to be used in child processes.
# logger_queue = init_logger(
#     stdout_lvl=None,
#     file_cfg=(logging.ERROR, error_log_path),
#     enqueue=True,
# )
# Init stdout logger
init_logger(reset_logger=False)

npy_files = []
for npy_prefix in options.npy_prefixes:
    # search inside the `YYYY-MM` directory
    str_month = f'{npy_prefix[:7]}'
    npy_files.extend(
        glob.glob(f'{options.src_base_dir}/{str_month}*/{npy_prefix}*.npy')
    )
    # search inside the `src_base_dir` directory
    npy_files.extend(glob.glob(f'{options.src_base_dir}/{npy_prefix}*.npy'))

if options.npy_list_file:
    with open(options.npy_list_file, 'r') as f:
        npy_list = f.readlines()
        # ignore lines that start with `#`
        npy_list = [npy.strip() for npy in npy_list if npy[0] != '#']
        npy_list = [f'{options.src_base_dir}/{npy}' for npy in npy_list]
        npy_files.extend(npy_list)

# npy_name -> npy_path
npy_files = {os.path.split(npy_file)[1]: npy_file for npy_file in npy_files}

if not options.override_output:
    exists_files = {}
    for npy_name, npy_file in npy_files.items():
        if os.path.exists(f'{options.output_dir}/{npy_name}'):
            exists_files[npy_name] = npy_file

    npy_names = npy_files.keys() - exists_files.keys()
    npy_files = {npy_name: npy_files[npy_name] for npy_name in npy_names}
    if exists_files:
        logging.warning(
            f'Skip {len(exists_files)} npy(s) (already generated), use the'
            f' `--override_output` option to regenerate again.'
        )


npy_files = list(npy_files.values())
if len(npy_files) == 0:
    logging.error(
        f'No matching npy is found for npy_prefixes={options.npy_prefixes} '
        f'or files specified in npy_list_file={options.npy_list_file}'
    )

args = [(configs, npy_file) for npy_file in npy_files]
configs, npy_path = args[0]

set_all_random_seed(0)
np.set_printoptions(precision=3, suppress=True)

vis = Visdom() if configs['remote_vis'] else None
out_dir_path = configs['output_base_path']
npy_filename = os.path.basename(npy_path)
npy_name = os.path.splitext(npy_filename)[0]
out_log_path = os.path.join(out_dir_path, f'{npy_name}.log')

# init_logger(
#     stdout_lvl=logging.INFO,
#     file_cfg=(logging.DEBUG, out_log_path),
#     logger_queue=logger_queue,
# )

npy_data = np.load(npy_path)

db_paths, _ = get_db_and_video_paths(npy_data, configs['db_base_path'])
db = DbReader(db_paths[0])
tname = str(npy_data['vehicle_info']['tname'])

s_time = configs['start_time']
duration = configs['duration']

sensors = npy_data['sensors']
cams_params = get_all_cameras_params(sensors)
main_cam_topic = CameraTopic.search_primary_driving_topic(cams_params.keys())
main_cam_params = cams_params[main_cam_topic]
db_configs = get_db_config(db)
radar_cfgs = get_all_radar_configs(db_configs, tname)
main_radar_topic = get_radar_topic(db)
device = torch.device('cuda')

try:
    db_s_time, duration = trim_db_time(
        db, s_time, duration, npy_data, look_ahead_dist=0
    )
except ValueError as e:
    # logging.error(f'skip {npy_name} due to {e}')
    exit()


obj_seg_model = SegmentationYolo(device=device)
e2emvm = E2emvm(multiview=False)

kj/filesystem-disk-unix.c++:1690: warning: PWD environment variable doesn't match current directory; pwd = /home/mars


Loaded SuperPoint model


In [2]:
def calc_color(track_count):
    """
    Calculate a color based on the track count.

    Parameters:
    - track_count (int): The input track count.

    Returns:
    - tuple: A color represented as (blue, green, red).
    """
    # Clamp track_count between 0 and 20
    track_count = max(0, min(track_count, 20))

    # Calculate ratio
    ratio = track_count / 30.0

    # Calculate blue and red channel values
    blue = int((1 - ratio) * 255)
    red = int(ratio * 255)

    # Return as a tuple (blue, green, red)
    return (blue, 0, red)


# Example usage
color = calc_color(15)
print(color)  # Example output: (170, 0, 85)

(127, 0, 127)


In [3]:
import logging
import traceback
import os
import cv2
import numpy as np
import torch
import networkx as nx
import itertools
from itertools import count
from typing import Dict, List, Optional
from lapsolver import solve_dense
from scipy.sparse import csr_matrix
from sknetwork.clustering import Leiden

from dataengine.generator.obstacle.debug_utils import draw_mask, Statistics
from dataengine.generator.obstacle.assignment import assign_segments
from dataengine.generator.obstacle.mask import (
    load_ego_masks,
    remove_ego_body_from_masks,
    remove_overlapped_area,
)

from marsneuralzoo.models.yolo_seg import SegmentationYolo
from marsneuralzoo.models.e2emvm import E2emvm
from marsdataio.dbhelper import (
    extract_image,
    collate_images,
    DbTopic,
    DbHandler,
)
from marsdataio.npyhelper import get_all_cameras_params
from marsdataio.videowriter import VideoWriter
from marsdataio.npyrenderer.renderer import generate_colors
from collections import defaultdict
from dataengine.generator.obstacle.assignment import assign_segments
from dataengine.generator.obstacle.segment_tracker import dfs, CameraTrackingHandler
from dataengine.generator.obstacle.object_solver import ObjectSolver

np.set_printoptions(precision=3, suppress=True)


import gc

for var in dir():
    if isinstance(globals()[var], torch.Tensor):
        del globals()[var]

torch.cuda.empty_cache()
gc.collect()

out_debug_dir = None

if configs['debug']:
    out_debug_dir = os.path.join(out_dir_path, npy_name)
    os.makedirs(out_debug_dir, exist_ok=True)

camera_tracking_handler = CameraTrackingHandler(
    npy_data=npy_data,
    obj_seg_model=obj_seg_model,
    e2emvm=e2emvm,
    cache_dir=configs['diskcache_dir'],
    ego_mask_dir=configs['ego_mask_dir'],
    out_debug_dir=out_debug_dir,
)

run_segmented_databases(db_paths, db_s_time, duration, camera_tracking_handler)

camera_tracking_handler_output = camera_tracking_handler.get_processed_data()

camera_tracking_handler.cleanup()

object_solver = ObjectSolver(
    npy_data=npy_data,
    camera_tracking_ouput=camera_tracking_handler_output,
    cache_dir=configs['diskcache_dir'],
    out_debug_dir=out_debug_dir,
)

Loading /media/vol/shared/obstacle/models/yolosegment/best.torchscript for TorchScript inference...


In [4]:
def prepare_undist(cam_params):
    out = []
    for cam in cam_params:

        K_undist = cv2.fisheye.estimateNewCameraMatrixForUndistortRectify(
            cam.K,
            cam.distort_coefs[:4],
            cam.img_wh,
            np.eye(3),
            balance=2.0,
        )
        map_x, map_y = cv2.fisheye.initUndistortRectifyMap(
            cam.K,
            cam.distort_coefs[:4],
            np.eye(3),
            K_undist,
            cam.img_wh,
            cv2.CV_16SC2,
        )
        out.append((K_undist, map_x, map_y))
    return out


class CameraImageHandler(DbHandler):
    def __init__(
        self,
        npy_data,
        cache_dir,
        out_debug_dir=None,
    ):
        """Handler to run model inference and tracking for all cameras."""
        self.npy_data = npy_data
        self._db_name = npy_data['db_filename']
        self._vehicle_name = npy_data['vehicle_info']['tname']
        self._cache_dir = cache_dir
        self._frame_count = 0

        # Camera initialization
        sensors = npy_data['sensors']
        self._cam_params = get_all_cameras_params(sensors)
        self._cam_param_list = list(self._cam_params.values())
        self.time_to_images = {}
        self.undist_params = prepare_undist(self._cam_param_list)
        self.time_to_undist_images = {}

        # debug option for intermideate results
        self.out_debug_dir = out_debug_dir

        if self.out_debug_dir is not None:
            video_width = -1
            video_height = -1

            self._video_pos_s = []
            for _, cam_param in self._cam_params.items():
                w, h = cam_param.video_pos + cam_param.img_wh
                video_width = max(video_width, w)
                video_height = max(video_height, h)
                self._video_pos_s.append(cam_param.video_pos)

            origin_video_path = os.path.join(self.out_debug_dir, f'origin.mp4')
            self._origin_video_writer = VideoWriter(
                origin_video_path, video_width, video_height, fps=20
            )

    def _handle_cam_msg(self, timestamp, seven_cam_image):
        imgs = []

        for _, cam_param in self._cam_params.items():
            img = extract_image(
                seven_cam_image, cam_param.img_wh, cam_param.video_pos
            )
            imgs.append(img)
        self.time_to_images[timestamp] = imgs

        undist_imgs = []
        for i in range(7):
            img_undist = cv2.remap(
                imgs[i],
                self.undist_params[i][1],
                self.undist_params[i][2],
                cv2.INTER_LINEAR,
            )
            undist_imgs.append(img_undist)
        self.time_to_undist_images[timestamp] = undist_imgs

        if self.out_debug_dir:
            frame = collate_images(imgs, self._video_pos_s)
            self._origin_video_writer.write(frame)

    def get_topics(self):
        return [DbTopic.CAM_MSG]

    def __call__(self, timestamp, topic, data):
        try:
            if topic == DbTopic.CAM_MSG:
                self._handle_cam_msg(timestamp, data)
                self._frame_count += 1

        except KeyboardInterrupt:
            raise KeyboardInterrupt
        except Exception as e:
            logging.error(
                f'Exception at {timestamp}, Topic: {topic}, Db: {self._db_name}'
                f'\n{traceback.format_exc()}'
            )
            raise e

    def cleanup(self):

        if self.out_debug_dir is not None:
            self._origin_video_writer.release()
            ts = list(self.time_to_images.keys())

            to_save = np.array(ts)
            np.save('saves_ns.npy', to_save)


camera_image_handler = CameraImageHandler(
    npy_data=npy_data,
    cache_dir=configs['diskcache_dir'],
    out_debug_dir=out_debug_dir,
)

run_segmented_databases(db_paths, db_s_time, duration, camera_image_handler)

camera_image_handler.cleanup()

In [5]:
from marstransform.lietrans import quat_rotate_np, make_se3_np, inv_se3_np
from marstransform.loctrans import sensor_to_body
from marsdataio.npyhelper import get_imu_params
import rerun as rr  # pip install rerun-sdk
import rerun.blueprint as rrb
import argparse
from scipy.spatial import cKDTree
from marstransform.lietrans import (
    quat_rotate_np,
    make_se3_np,
    inv_se3_np,
    expm_so3_np
)
from dataengine.generator.obstacle.object_solver import apply_se3

my_blueprint = rrb.Blueprint(
    rrb.Horizontal(
        rrb.Spatial3DView(
                name='3D',
                origin='world/body',
                contents=['world/**']
            ),
        rrb.Horizontal(
            rrb.Vertical(
                
                rrb.Spatial2DView(
                    name="cam0",
                    origin='world/body/cam0',
                    contents=['world/body/cam0/bgr', 'world/obstacles/**'],
                ),
                rrb.Spatial2DView(
                    name="cam1",
                    origin='world/body/cam1',
                    contents=['world/body/cam1/bgr', 'world/obstacles/**'],
                ),
                rrb.Spatial2DView(
                    name="cam2",
                    origin='world/body/cam2',
                    contents=['world/body/cam2/bgr', 'world/obstacles/**'],
                ),
            ),
            rrb.Vertical(
                rrb.Spatial2DView(
                    name="cam3",
                    origin='world/body/cam3',
                    contents=['world/body/cam3/bgr', 'world/obstacles/**'],
                ),
                rrb.Spatial2DView(
                    name="cam5",
                    origin='world/body/cam5',
                    contents=['world/body/cam5/bgr', 'world/obstacles/**'],
                ),
                rrb.Spatial2DView(
                    name="cam4",
                    origin='world/body/cam4',
                    contents=['world/body/cam4/bgr', 'world/obstacles/**'],
                ),
                rrb.Spatial2DView(
                    name="cam6",
                    origin='world/body/cam6',
                    contents=['world/body/cam6/bgr', 'world/obstacles/**'],
                ),
            )
        )
    )
    
)

rr.init("hi", spawn=True)
rr.send_blueprint(my_blueprint)

rr.log(
    "world", rr.ViewCoordinates.RIGHT_HAND_Z_DOWN, static=True
)  # Set an up-axis

rr.log(
    "world/xyz",
    rr.Arrows3D(
        vectors=[[10, 0, 0], [0, 10, 0], [0, 0, 10]],
        colors=[[255, 0, 0], [0, 255, 0], [0, 0, 255]],
    ),
    static=True,
)

T_we = next(iter(object_solver.T_ebs.values()))
T_we = inv_se3_np(T_we)


undist_param = camera_image_handler.undist_params

w, h = object_solver._cam_params[0].img_wh
dist = 1.0

for time, T_eb in object_solver.T_ebs.items():
    rr.set_time_seconds('time', time)

    T_wb = T_we @ T_eb

    rr.log(
        'world/body',
        rr.Transform3D(
            translation=T_wb[:3, 3], mat3x3=T_wb[:3, :3], axis_length=1
        ),
    )

    rr.log(
        "world/body/point",
        rr.Points3D([0, 0, 0], radii=0.50, colors=[255, 200, 10]),
    )
    undist_images = camera_image_handler.time_to_undist_images[time]
    for i, T_bc in enumerate(object_solver.T_bcs):
        rr.log(
            f'world/body/cam{i}',
            rr.Transform3D(translation=T_bc[:3, 3], mat3x3=T_bc[:3, :3]),
        )
        rr.log(
            f'world/body/cam{i}',
            rr.Pinhole(
                image_from_camera=undist_param[i][0],
                resolution=[w, h],
                image_plane_distance=dist,
            ),
        )

        rr.log(
            f'world/body/cam{i}/bgr',
            rr.Image(undist_images[i], color_model="BGR").compress(
                jpeg_quality=95
            ),
        )


timestamps_in_scope = np.array(list(object_solver.T_ems.keys()))
T_ems = np.array(list(object_solver.T_ems.values()))
t_ems = T_ems[:, 3, :3]
ego_pos_tree = cKDTree(t_ems)

wlh_prior = object_solver._wlh_prior
half_xyz = wlh_prior[[1, 0, 2]] / 2

for op_id, mea in object_solver.ob_id_to_mea.items():
    if len(mea.timestamps) < 2:
        continue

    for i in range(len(mea.timestamps)):

        time = mea.timestamps[i]
        t_eo = mea.t_eos[i]
        yaw_mo = mea.yaw_mos[i]
        if np.isnan(t_eo).all():
            continue

        _, idx = ego_pos_tree.query(t_eo)
        ref_time = timestamps_in_scope[idx]
        T_em = object_solver.T_ems[ref_time]
        T_me = inv_se3_np(T_em)

        # orientation
        R_mo = expm_so3_np(np.array([0, 0, yaw_mo]))
        R_wo = T_we[:3, :3] @ R_mo

        # center
        t_mo = apply_se3(t_eo, T_me)
        t_mo[2] += half_xyz[2]
        t_eo = apply_se3(t_mo, T_em)
        t_wo = apply_se3(t_eo, T_we)

        rr.set_time_seconds('time', time)
        rr.log(
            f'world/obstacles/ob{op_id}/box',
            rr.Boxes3D(
                centers=t_wo,
                half_sizes=half_xyz,
                rotations=R_wo,
                class_ids=op_id,
            ),
        ),

    rr.set_time_seconds('time', mea.timestamps[-1] + 0.02)
    rr.log(f'world/obstacles/ob{op_id}/box', rr.Clear(recursive=False))

[2024-12-05T03:45:37Z INFO  re_sdk::spawn] A process is already listening at this address. Assuming it's a Rerun Viewer. addr=0.0.0.0:9876


[2024-12-05 12:45:42,269] [WARNING] boxes3d_ext.py: Could not infer the type of rotation from the input. Please use `quaternions` or `rotation_axis_angles`.
